In [1]:
import sys, streamlit, pandas, requests, sqlalchemy, plotly
from dotenv import load_dotenv
import os

load_dotenv()
print("Python:", sys.version.split()[0])
print("Streamlit:", streamlit.__version__)
print("Pandas:", pandas.__version__)
print("SQLAlchemy:", sqlalchemy.__version__)
print("DB URL loaded:", os.getenv("DATABASE_URL"))

Python: 3.13.14
Streamlit: 1.65.0
Pandas: 3.0.6
SQLAlchemy: 2.1.4
DB URL loaded: None


In [5]:
import os, json, time, requests
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.getenv("API_BASE_URL")
headers = {
    "x-rapidapi-key": os.getenv("RAPIDAPI_KEY"),
    "x-rapidapi-host": os.getenv("RAPIDAPI_HOST"),
}


def fetch_json(path, params=None, retries=3, backoff=3):
    """GET an API path with retries; returns parsed JSON or None."""
    url = f"{BASE_URL}{path}"
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(
                url, headers=headers, params=params,
                timeout=(10, 60),  # (connect, read) seconds
            )
            print(f"Attempt {attempt}: status {resp.status_code}")
            if resp.status_code == 200:
                return resp.json()
            if resp.status_code in (401, 403, 404):
                print(resp.text[:300])
                return None  # retrying will not help
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt} failed: {type(e).__name__}")
        time.sleep(backoff * attempt)
    return None


data = fetch_json("/matches/v1/live")
if data:
    print("Top-level keys:", list(data.keys()))
    os.makedirs("../data", exist_ok=True)
    with open("../data/live_raw.json", "w") as f:
        json.dump(data, f)  # cache so we don't re-call the API

Attempt 1: status 200
Top-level keys: ['typeMatches', 'filters', 'appIndex', 'responseLastUpdated']


In [6]:
import json
import pandas as pd

with open("../data/live_raw.json") as f:
    data = json.load(f)

# 1) Inspect the structure first
first_type = data["typeMatches"][0]
print("matchType:", first_type.get("matchType"))
print("keys in a typeMatches block:", list(first_type.keys()))

# 2) Flatten into rows
rows = []
for type_block in data.get("typeMatches", []):
    match_type = type_block.get("matchType")
    for series_block in type_block.get("seriesMatches", []):
        wrapper = series_block.get("seriesAdWrapper")
        if not wrapper:  # ad blocks have no wrapper
            continue
        for m in wrapper.get("matches", []):
            info = m.get("matchInfo", {})
            rows.append({
                "match_id": info.get("matchId"),
                "match_type": match_type,
                "series": wrapper.get("seriesName"),
                "match_desc": info.get("matchDesc"),
                "format": info.get("matchFormat"),
                "team1": info.get("team1", {}).get("teamName"),
                "team2": info.get("team2", {}).get("teamName"),
                "venue": info.get("venueInfo", {}).get("ground"),
                "city": info.get("venueInfo", {}).get("city"),
                "state": info.get("state"),
                "status": info.get("status"),
            })

live_df = pd.DataFrame(rows)
print(len(live_df), "live matches found")
live_df.head(10)

matchType: International
keys in a typeMatches block: ['matchType', 'seriesMatches']
5 live matches found


,match_id,match_type,series,match_desc,format,team1,team2,venue,city,state,status
0,151602,International,"West Indies tour of India, 2026",5th T20I,T20,India,West Indies,M.Chinnaswamy Stadium,Bengaluru,Delay,Toss delayed due to rain
1,169950,International,ICC Cricket World Cup League Two 2023-27,138th Match,ODI,Canada,Nepal,Al Amerat Cricket Ground (Ministry Turf 1),Al Amerat,Complete,Canada won by 24 runs
2,172677,International,"ODI Tri-Nation Series in UAE, 2026",1st Match,ODI,Zimbabwe,Bangladesh,Sheikh Zayed Stadium,Abu Dhabi,In Progress,Zimbabwe opt to bat
3,176441,Women,"ECA Women's European Cup, 2026",6th Match,T20,Romania Women,Sweden Women,"Moara Vlasiei Cricket Ground, Bucharest",Ilfov County,In Progress,Romania Women opt to bat
4,176430,Women,"ECA Women's European Cup, 2026",5th Match,T20,Romania Women,Austria Women,"Moara Vlasiei Cricket Ground, Bucharest",Ilfov County,Complete,Austria Women won by 7 wkts
